# Extraction Analysis

Recovery (recall/precision vs. ground truth) and hallucination rate for a single extraction run.

In [15]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'pond'
MODEL           = 'gemma-3-27b'
EXTRACTION_DATE = '2026_04_19'  # set to None to use most recent run

In [17]:
import pandas as pd
from analysis.loaders import load_extraction, load_combined_judgements, load_ground_truth
from scholarlm.utils.unit_conversion import apply_unit_conversion
from analysis.metrics import recovery_rate, hallucination_rate, per_paper_metrics
from analysis.plots import recovery_bar, probability_distribution
from experiments.run_extraction import load_dataset_config

config = load_dataset_config(DATASET)

In [18]:
config.ground_truth_file

'data/pond/ground_truth.csv'

## Load data

In [4]:
records = load_extraction(DATASET, MODEL, EXTRACTION_DATE)
extraction_df = pd.DataFrame(records)
extraction_df = apply_unit_conversion(extraction_df, config.unit_conversion_table)
print(f'{len(extraction_df)} extracted measurements')
extraction_df.head()

11768 extracted measurements


,title,author,year,paper_code,document_id,name,abbreviations,location,ecosystem,entity_id,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,converted_value
0,agricultural freshwater pond supports diverse ...,chopyk et al.,2018,agricultural_freshwater,0,agricultural pond,NaN,Mid-Atlantic United States,pond,doc_0_entity_0,...,0.26,ha,[1],[None],[None],[None],[text],"[standard, ponds are generally defined as smal...",0,2600.00
1,agricultural freshwater pond supports diverse ...,chopyk et al.,2018,agricultural_freshwater,0,agricultural pond,NaN,Mid-Atlantic United States,pond,doc_0_entity_0,...,0.26,ha,[1],[None],[None],[None],[text],"[standard, ponds are generally defined as smal...",1,2600.00
2,agricultural freshwater pond supports diverse ...,chopyk et al.,2018,agricultural_freshwater,0,agricultural pond,NaN,Mid-Atlantic United States,pond,doc_0_entity_0,...,3.35,meters,[1],[None],[None],[None],[text],"[standard, ponds are generally defined as smal...",2,3.35
3,agricultural freshwater pond supports diverse ...,chopyk et al.,2018,agricultural_freshwater,0,agricultural pond,NaN,Mid-Atlantic United States,pond,doc_0_entity_0,...,3.35,m,[1],[None],[None],[None],[text],"[standard, ponds are generally defined as smal...",3,3.35
4,agricultural freshwater pond supports diverse ...,chopyk et al.,2018,agricultural_freshwater,0,agricultural pond,NaN,Mid-Atlantic United States,pond,doc_0_entity_0,...,3.35,meters,[1],[None],[None],[None],[text],"[standard, ponds are generally defined as smal...",4,3.35


In [5]:
ground_truth_df = load_ground_truth(config)
print(f'{len(ground_truth_df)} ground truth measurements')
ground_truth_df.head()

FileNotFoundError: Ground truth file not found: data/pond/ground_truth.csv

In [19]:
import os

# Get the current working directory
cwd = os.getcwd()

# Print the directory
print(cwd)

/projectnb/mcnet/kevin/coastal/scholarlm/analysis


In [ ]:
judgements = load_combined_judgements(DATASET, MODEL, EXTRACTION_DATE)
judged_df = pd.DataFrame(judgements)
print(f'{len(judged_df)} judged measurements')

## Recovery

In [ ]:
# Update strict_matching to match your dataset's entity key columns
STRICT_MATCHING = {'entity': ['name', 'location']}
FUZZY_MATCHING  = {'value': ['value']}   # or {} for strict-only

stats = recovery_rate(
    extraction_df,
    ground_truth_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
print(stats)

## Hallucination rate

In [ ]:
hall = hallucination_rate(judged_df)
print(hall)

fig = probability_distribution(judged_df, prob_col='judgement_p_true', label_col='judgement_combined')
fig.savefig('figures/prob_distribution.pdf', bbox_inches='tight')
fig.show()

## Per-paper breakdown

In [ ]:
paper_df = per_paper_metrics(
    extraction_df,
    ground_truth_df,
    judged_df=judged_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
paper_df.sort_values('recall').round(3)